# CSV to Excel Studio (Python version)

A notebook port of the browser tool. It will:

1. **Clean**: trim spaces, drop empty rows, drop duplicates, detect numbers and dates (leading zeros like `007` stay text)
2. **Shape**: rename, hide and retype columns, change header style
3. **Export**: Excel (auto-fit widths, filter buttons), CSV or JSON, plus an activity log

Run the cells top to bottom. Change settings only in **cell 2**.

In [1]:
# Cell 1: setup. Uncomment the next line if packages are missing.
# %pip install pandas openpyxl
import csv, io, json, re, logging
from datetime import datetime
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from IPython.display import display

LOGS = []
class _ListHandler(logging.Handler):
    def emit(self, record):
        LOGS.append(f"{datetime.now():%Y-%m-%d %H:%M:%S} [{record.levelname}] {record.getMessage()}")

log = logging.getLogger("csv_studio")
log.setLevel(logging.INFO)
log.handlers = [_ListHandler()]
print("Ready.")

Ready.


In [2]:
# Cell 2: settings (edit these)

INPUT_FILE   = "customers.csv"   # path to your CSV/TSV/TXT file
USE_SAMPLE   = True              # True = ignore INPUT_FILE and use the built-in messy sample

# Step 1: clean up
FIRST_ROW_IS_HEADER = True
TRIM_SPACES         = True
REMOVE_EMPTY_ROWS   = True
REMOVE_DUPLICATES   = False      # keeps the first copy of each row
DETECT_TYPES        = True       # numbers and dates (leading zeros stay text)
DELIMITER           = None       # None = auto-detect, or "," ";" "\t" "|"
ENCODING            = "utf-8"    # or "windows-1252", "iso-8859-1"

# Step 2: columns
HEADER_CASE    = "as"            # "as" | "title" | "snake" | "upper"
COLUMN_RENAMES = {}              # e.g. {"cust": "Customer"}   (keys = header after HEADER_CASE is applied)
DROP_COLUMNS   = []              # e.g. ["notes"]
TYPE_OVERRIDES = {}              # e.g. {"id": "text", "amount": "number", "signup_date": "date"}

# Step 3: export
OUTPUT_FORMAT = "xlsx"           # "xlsx" | "csv" | "json"
OUTPUT_NAME   = "cleaned_data"   # file name without extension
SHEET_NAME    = "Sheet1"
ADD_FILTER    = True             # filter buttons on the header (xlsx)
AUTO_FIT      = True             # auto-fit column widths (xlsx)

In [3]:
# Cell 3: load the file

SAMPLE = (
    "id,customer,city,signup_date,amount,notes\n"
    "001,  Asha Rao ,Hubballi,2026-01-14,\"$1,200.50\",paid\n"
    "002,Vikram Shah,Pune,2026-02-03,980,pending\n"
    "\n"
    "002,Vikram Shah,Pune,2026-02-03,980,pending\n"
    "003,Meera Iyer ,Bengaluru,2026-02-19,\"$2,450.00\",\n"
    "004,  Rohan Das,Delhi,2026-03-07,310.75,paid\n"
    "005,Sana Khan,Mumbai,,1500,paid\n"
    ",,,,,\n"
    "006,Dev Patel,Ahmedabad,2026-04-22,\"$75\",refund\n"
)

def read_text(path, encoding):
    raw = Path(path).read_bytes()
    if not raw:
        raise ValueError(f'"{path}" is empty (0 bytes).')
    if raw[:2] == b"PK":
        raise ValueError(f'"{path}" looks like an Excel or zip file, not a CSV.')
    if raw[:2] == b"\xff\xfe":
        encoding = "utf-16"
    text = raw.decode(encoding, errors="replace").lstrip("\ufeff")
    if not text.strip():
        raise ValueError("The file contains only blank space.")
    if "\x00" in text:
        raise ValueError("The file contains binary data, so it is probably not a text CSV.")
    if "\ufffd" in text:
        log.warning("Odd symbols found. Try ENCODING = 'windows-1252'.")
    return text

def detect_delimiter(text):
    if DELIMITER:
        return DELIMITER
    try:
        return csv.Sniffer().sniff(text[:5000], delimiters=",;\t|").delimiter
    except csv.Error:
        return ","

if USE_SAMPLE:
    SOURCE_NAME, text = "sample-customers.csv", SAMPLE
else:
    SOURCE_NAME, text = Path(INPUT_FILE).name, read_text(INPUT_FILE, ENCODING)

delim = detect_delimiter(text)
raw_rows = list(csv.reader(io.StringIO(text), delimiter=delim))
while raw_rows and raw_rows[-1] in ([], [""]):
    raw_rows.pop()                      # trailing blank line
width = max((len(r) for r in raw_rows), default=0)
if width == 0:
    raise ValueError("No columns were found in this file.")
raw_rows = [r + [""] * (width - len(r)) for r in raw_rows]

names = {",": "Comma", ";": "Semicolon", "\t": "Tab", "|": "Pipe"}
log.info(f'Loaded "{SOURCE_NAME}": {len(raw_rows)} rows, {width} columns, {names.get(delim, delim)} separated.')
print(f"{SOURCE_NAME}: {len(raw_rows)} lines, {width} columns, separator = {names.get(delim, repr(delim))}")
if width == 1 and ";" in text:
    print("Tip: only one column found. Try DELIMITER = ';'")

INFO:csv_studio:Loaded "sample-customers.csv": 10 rows, 6 columns, Comma separated.


sample-customers.csv: 10 lines, 6 columns, separator = Comma


In [4]:
# Cell 4: step 1 - clean up

def clean(rows):
    if TRIM_SPACES:
        rows = [[str(c).strip() for c in r] for r in rows]
    header = rows[0] if (FIRST_ROW_IS_HEADER and rows) else None
    body = rows[1:] if header is not None else rows

    n_blank = sum(all(c == "" for c in r) for r in body)
    if REMOVE_EMPTY_ROWS:
        body = [r for r in body if not all(c == "" for c in r)]

    seen, n_dupe_all = set(), 0
    for r in body:
        k = tuple(r)
        n_dupe_all += k in seen
        seen.add(k)
    if REMOVE_DUPLICATES:
        seen, kept = set(), []
        for r in body:
            k = tuple(r)
            if k not in seen:
                seen.add(k)
                kept.append(r)
        body = kept

    if header is None:
        header = [""] * len(rows[0])
    header = [h if h != "" else f"Column {i + 1}" for i, h in enumerate(header)]
    return header, body, n_blank, n_dupe_all

orig_header, body, n_blank, n_dupe = clean(raw_rows)

def case_header(s, mode):
    if mode == "title":
        return re.sub(r"(^|[\s_-])(\w)", lambda m: (" " if m.group(1) else "") + m.group(2).upper(), s.lower()).strip()
    if mode == "snake":
        return re.sub(r"[^a-z0-9]+", "_", s.strip().lower()).strip("_")
    if mode == "upper":
        return s.upper()
    return s

print(f"Rows kept: {len(body)}   |   empty rows found: {n_blank}   |   duplicate rows found: {n_dupe}")
if n_blank and not REMOVE_EMPTY_ROWS:
    print("Tip: set REMOVE_EMPTY_ROWS = True")
if n_dupe and not REMOVE_DUPLICATES:
    print("Tip: set REMOVE_DUPLICATES = True")
log.info(f"Cleaned: {len(body)} rows kept, {n_blank} empty and {n_dupe} duplicate rows found.")

INFO:csv_studio:Cleaned: 7 rows kept, 2 empty and 1 duplicate rows found.


Rows kept: 7   |   empty rows found: 2   |   duplicate rows found: 1
Tip: set REMOVE_DUPLICATES = True


In [5]:
# Cell 5: step 2 - columns and types

NUM_RE = re.compile(r"^[-+]?\(?[$₹€£]?\s?\d{1,3}(,\d{3})*(\.\d+)?\)?$|^[-+]?[$₹€£]?\s?\d+(\.\d+)?$")
ISO_RE = re.compile(r"^(\d{4})-(\d{1,2})-(\d{1,2})([ T][\d:]+)?$")
DMY_RE = re.compile(r"^(\d{1,2})([/.-])(\d{1,2})\2(\d{4})$")

def is_num(v):
    return bool(NUM_RE.match(v)) and not re.match(r"^0\d", v.lstrip("-+$₹€£ "))

def to_num(v):
    neg = v.startswith("(") or v.startswith("-")
    digits = re.sub(r"[^\d.]", "", v)
    if not digits:
        return None
    n = float(digits)
    n = -n if neg else n
    return int(n) if n.is_integer() and "." not in digits else n

def to_date(v):
    try:
        m = ISO_RE.match(v)
        if m:
            return datetime(int(m[1]), int(m[2]), int(m[3]))
        m = DMY_RE.match(v)
        if m:
            return datetime(int(m[4]), int(m[3]), int(m[1]))   # day first
    except ValueError:
        pass
    return None

def is_date(v):
    return to_date(v) is not None

def excel_date_format(v):
    m = DMY_RE.match(v)
    if m:
        sep = m[2]
        return f"{'d' if len(m[1]) == 1 else 'dd'}{sep}{'m' if len(m[3]) == 1 else 'mm'}{sep}yyyy"
    return "yyyy-mm-dd"

# build column table
cols = []
used = set()
for i, h in enumerate(orig_header):
    name = case_header(h, HEADER_CASE)
    name = COLUMN_RENAMES.get(name, name) or f"Column {i + 1}"
    base, k = name, 2
    while name in used:
        name, k = f"{base}_{k}", k + 1
    used.add(name)

    vals = [r[i] for r in body if r[i] != ""]
    has_lead0 = any(re.match(r"^0\d", v) for v in vals)
    auto = "text"
    if DETECT_TYPES and vals and not has_lead0:
        if all(is_num(v) for v in vals):
            auto = "number"
        elif all(is_date(v) for v in vals):
            auto = "date"
    ctype = TYPE_OVERRIDES.get(name, auto)
    cols.append({
        "i": i, "name": name, "auto": auto, "type": ctype,
        "include": name not in DROP_COLUMNS and h not in DROP_COLUMNS,
        "empty": len(body) - len(vals), "unique": len(set(vals)),
        "date_fmt": excel_date_format(vals[0]) if ctype == "date" and vals else "yyyy-mm-dd",
    })

def convert(v, t):
    if v == "":
        return None
    if t == "number":
        n = to_num(v)
        return v if n is None else n
    if t == "date":
        d = to_date(v)
        return v if d is None else d
    return v

out_cols = [c for c in cols if c["include"]]
if not out_cols:
    raise ValueError("Every column is hidden. Remove some names from DROP_COLUMNS.")
rows_out = [[convert(r[c["i"]], c["type"]) for c in out_cols] for r in body]
df = pd.DataFrame(rows_out, columns=[c["name"] for c in out_cols])

summary = pd.DataFrame([{
    "column": c["name"], "included": c["include"], "detected": c["auto"], "used_as": c["type"],
    "unique": c["unique"], "empty": c["empty"]} for c in cols])
display(summary)
log.info(f"Columns ready: {len(out_cols)} of {len(cols)} included.")

,column,included,detected,used_as,unique,empty
0,id,True,text,text,6,0
1,customer,True,text,text,6,0
2,city,True,text,text,6,0
3,signup_date,True,date,date,5,1
4,amount,True,number,number,6,0
5,notes,True,text,text,3,1


INFO:csv_studio:Columns ready: 6 of 6 included.


In [6]:
# Cell 6: live preview (first 100 rows). Change SEARCH or SORT_BY to explore.

SEARCH  = ""        # e.g. "pune" filters rows containing this text
SORT_BY = None      # e.g. "amount"
DESCENDING = False

view = df.copy()
if SEARCH:
    mask = view.astype(str).apply(lambda col: col.str.contains(SEARCH, case=False, na=False)).any(axis=1)
    view = view[mask]
if SORT_BY:
    view = view.sort_values(SORT_BY, ascending=not DESCENDING, na_position="last")

print(f"{len(view):,} row(s). Empty cells show as None/NaT/NaN. Export includes every row.")
display(view.head(100))
display(df.dtypes.rename("dtype").to_frame().T)

7 row(s). Empty cells show as None/NaT/NaN. Export includes every row.


,id,customer,city,signup_date,amount,notes
0,001,Asha Rao,Hubballi,2026-01-14,1200.50,paid
1,002,Vikram Shah,Pune,2026-02-03,980.00,pending
2,002,Vikram Shah,Pune,2026-02-03,980.00,pending
3,003,Meera Iyer,Bengaluru,2026-02-19,2450.00,None
4,004,Rohan Das,Delhi,2026-03-07,310.75,paid
5,005,Sana Khan,Mumbai,NaT,1500.00,paid
6,006,Dev Patel,Ahmedabad,2026-04-22,75.00,refund


,id,customer,city,signup_date,amount,notes
dtype,object,object,object,datetime64[ns],float64,object


In [7]:
# Cell 7: step 3 - export

def safe_name(s):
    return re.sub(r'[\\/:*?"<>|]+', "_", s or "data")

def fmt_cell(v):
    if v is None:
        return ""
    if isinstance(v, datetime):
        return v.strftime("%Y-%m-%d")
    return str(v)

def export_xlsx(path):
    wb = Workbook()
    ws = wb.active
    ws.title = re.sub(r"[\\/?*\[\]:]", "_", SHEET_NAME or "Sheet1")[:31]
    ws.append([c["name"] for c in out_cols])
    for r in rows_out:
        ws.append(r)
    for k, c in enumerate(out_cols, start=1):
        if c["type"] == "date":
            for cell in ws.iter_cols(min_col=k, max_col=k, min_row=2, max_row=ws.max_row).__next__():
                if isinstance(cell.value, datetime):
                    cell.number_format = c["date_fmt"]
        if AUTO_FIT:
            longest = max([len(c["name"])] + [len(fmt_cell(r[k - 1])) for r in rows_out])
            width = min(max(longest + 2, 8), 60)
            if c["type"] == "date":
                width = max(width, 14)
        else:
            width = 14 if c["type"] == "date" else 9
        ws.column_dimensions[get_column_letter(k)].width = width
    if ADD_FILTER and rows_out:
        ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = "A2"
    wb.save(path)

def export_csv(path):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow([c["name"] for c in out_cols])
        for r in rows_out:
            w.writerow([fmt_cell(v) for v in r])

def export_json(path):
    recs = []
    for r in rows_out:
        recs.append({c["name"]: (v.strftime("%Y-%m-%d") if isinstance(v, datetime) else v)
                     for c, v in zip(out_cols, r)})
    Path(path).write_text(json.dumps(recs, indent=2, ensure_ascii=False), encoding="utf-8")

fmt = OUTPUT_FORMAT.lower()
if fmt not in ("xlsx", "csv", "json"):
    raise ValueError("OUTPUT_FORMAT must be 'xlsx', 'csv' or 'json'.")
out_path = Path(f"{safe_name(OUTPUT_NAME)}.{fmt}")
try:
    {"xlsx": export_xlsx, "csv": export_csv, "json": export_json}[fmt](out_path)
    log.info(f"Saved {out_path} with {len(rows_out):,} rows.")
    print(f"Saved {out_path.resolve()} with {len(rows_out):,} rows.")
except Exception as e:
    log.error(f"Export failed: {e}")
    raise

# In Google Colab, uncomment to download:
# from google.colab import files; files.download(str(out_path))

INFO:csv_studio:Saved cleaned_data.xlsx with 7 rows.


Saved /content/cleaned_data.xlsx with 7 rows.


In [8]:
# Cell 8: activity log (also saved to a text file)
log_text = "\n".join(LOGS) or "No activity yet."
print(log_text)
Path("csv-to-excel-log.txt").write_text(log_text, encoding="utf-8")

2026-11-02 07:58:43 [INFO] Loaded "sample-customers.csv": 10 rows, 6 columns, Comma separated.
2026-11-02 07:58:48 [INFO] Cleaned: 7 rows kept, 2 empty and 1 duplicate rows found.
2026-11-02 07:58:51 [INFO] Columns ready: 6 of 6 included.
2026-11-02 07:59:04 [INFO] Saved cleaned_data.xlsx with 7 rows.


302

### Quick reference

| Browser tool option | Notebook setting |
|---|---|
| First row is the header | `FIRST_ROW_IS_HEADER` |
| Trim / empty rows / duplicates | `TRIM_SPACES`, `REMOVE_EMPTY_ROWS`, `REMOVE_DUPLICATES` |
| Detect numbers and dates | `DETECT_TYPES` |
| Separator / encoding | `DELIMITER`, `ENCODING` |
| Header style | `HEADER_CASE` |
| Rename / hide / retype columns | `COLUMN_RENAMES`, `DROP_COLUMNS`, `TYPE_OVERRIDES` |
| Format, file name, sheet name | `OUTPUT_FORMAT`, `OUTPUT_NAME`, `SHEET_NAME` |
| Filter buttons, auto-fit | `ADD_FILTER`, `AUTO_FIT` |

To use your own file, set `USE_SAMPLE = False` and `INPUT_FILE = "yourfile.csv"`, then re-run from cell 2.